# BADAS-Open layers
Lists the parts of BADAS-Open and every Linear layer (the layers LoRA can attach to). Runs on the CPU.
Needs the weights in `C:\safe-distance-data\models\badas-open` (`scripts/get_badas_source.py --weights`).

In [ ]:
# Cell 1: load BADAS-Open on the CPU (no GPU needed)
import re
import sys
from collections import defaultdict
from pathlib import Path

import torch

BADAS = Path(r"C:\safe-distance-data\models\badas-open")   # downloaded by scripts/get_badas_source.py --weights
sys.path.insert(0, str(BADAS / "src"))
from models.vjepa import VJEPAModel  # noqa: E402

badas = VJEPAModel(model_name="facebook/vjepa2-vitl-fpc16-256-ssv2",
                   checkpoint_path=str(BADAS / "weights" / "badas_open.pth"),
                   device="cpu", frame_count=16, img_size=224, target_fps=8.0, use_sliding_window=False)
badas.load()
net = badas.model
print(type(net).__name__)

In [ ]:
# Cell 2: the big parts, with their parameter counts
def count(m):
    return sum(p.numel() for p in m.parameters())

for name, child in net.named_children():
    print(f"{name:40s} {type(child).__name__:30s} {count(child) / 1e6:8.2f} M")
    for n2, c2 in child.named_children():
        print(f"    {name}.{n2:34s} {type(c2).__name__:30s} {count(c2) / 1e6:8.2f} M")

In [ ]:
# Cell 3: every Linear layer (what LoRA can attach to), grouped: layer numbers replaced by N
groups = defaultdict(list)
for name, m in net.named_modules():
    if isinstance(m, torch.nn.Linear):
        groups[re.sub(r"\.\d+\.", ".N.", name)].append((name, m))
print(f"{'pattern':70s} {'count':>5s}  in -> out")
for pat, items in groups.items():
    m = items[0][1]
    print(f"{pat:70s} {len(items):5d}  {m.in_features} -> {m.out_features}")

In [ ]:
# Cell 4: the full list of Linear layer names (exact names to put in LoRA's target_modules)
for name, m in net.named_modules():
    if isinstance(m, torch.nn.Linear):
        print(name, m.in_features, "->", m.out_features)